In [ ]:
!pip install monai itk SimpleITK einops nibabel torch

  Using cached monai-1.5.2-py3-none-any.whl.metadata (13 kB)
  Using cached itk-5.4.6-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached simpleitk-2.5.5-cp311-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.4 kB)
  Using cached itk_core-5.4.6-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached itk_numerics-5.4.6-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached itk_io-5.4.6-cp311-abi3-manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached itk_filtering-5.4.6-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached itk_registration-5.4.6-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
  Using cached itk_segmentation-5.4.6-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (22 kB)
Using cached monai-1.5.2-py3-none-any.whl (2.7 MB)
Using cached itk-5.4.6-cp311-abi3-manylinux_2_28_x86_64.whl (16 k

In [ ]:
import os
import torch
import monai
import monai.transforms as mt
import SimpleITK as sitk
import numpy as np
from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [ ]:
drive.mount('/content/drive')

UNLABELED_DIR = "/content/drive/MyDrive/panther/ImagesTr_unlabeled"
PSEUDO_LABELS_DIR = "/content/drive/MyDrive/pseudo_labels_SwinUNETR"
PSEUDO_WEIGHTS_DIR = "/content/drive/MyDrive/pseudo_weights_SwinUNETR"
MODEL_PATH = "/content/drive/MyDrive/SwinUNETR_models/SwinUNETR_random_phase2_v2.pth"
os.makedirs(PSEUDO_LABELS_DIR, exist_ok=True)
os.makedirs(PSEUDO_WEIGHTS_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
class PseudoLabelingDataSet(monai.data.Dataset):
    def __init__(self, imagesTr_unlabeled):
        self.image_paths_unlabeled = sorted(Path(imagesTr_unlabeled).glob("*.mha"))
        data = [{"image": str(path)} for path in self.image_paths_unlabeled]
        transforms = mt.Compose([
            mt.LoadImaged(keys=["image"], reader="ITKReader"),
            mt.EnsureChannelFirstd(keys=["image"]),
            mt.Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
            mt.Orientationd(keys=["image"], axcodes="RAS"),
            mt.NormalizeIntensityd(keys=["image"]),
        ])
        super().__init__(data=data, transform=transforms)

In [ ]:
def predict_tta(model, image, device):

    def infer(img):
        return sliding_window_inference(
            img, roi_size=(96, 96, 96), sw_batch_size=2,
            predictor=model, overlap=0.5
        )

    preds = []

    with torch.no_grad():

        preds.append(torch.softmax(infer(image), dim=1))

        preds.append(torch.softmax(infer(image.flip(2)), dim=1).flip(2))

        preds.append(torch.softmax(infer(image.flip(3)), dim=1).flip(3))

        preds.append(torch.softmax(infer(image.flip(4)), dim=1).flip(4))

        preds.append(torch.softmax(infer(image.flip(2).flip(3)), dim=1).flip(3).flip(2))

        preds.append(torch.softmax(infer(image.flip(2).flip(4)), dim=1).flip(4).flip(2))

        preds.append(torch.softmax(infer(image.flip(3).flip(4)), dim=1).flip(4).flip(3))

        preds.append(torch.softmax(infer(image.flip(2).flip(3).flip(4)), dim=1).flip(4).flip(3).flip(2))

    return torch.stack(preds).mean(0)


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SwinUNETR(in_channels=1, out_channels=3, feature_size=48, spatial_dims=3).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device)
if any(k.startswith('module.') for k in state_dict.keys()):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
print("Model loaded!")

dataset = PseudoLabelingDataSet(UNLABELED_DIR)
print(f"Founded {len(dataset)} unlabeled images")

min_confidence = 0.7
accepted = 0
rejected = 0

for i, data in enumerate(tqdm(dataset)):
    image = data["image"].unsqueeze(0).to(device)
    img_path = dataset.image_paths_unlabeled[i]

    probs = predict_tta(model, image, device)
    tumor_prob = probs[0, 1].cpu().numpy()
    pancreas_prob = probs[0, 2].cpu().numpy()
    pred = torch.argmax(probs[0], dim=0).cpu().numpy().astype(np.uint8)

    tumor_mask = pred == 1

    if tumor_mask.sum() == 0:
        rejected += 1
        continue

    mean_tumor_confidence = tumor_prob[tumor_mask].mean()
    print(f"Mean confidence tumor: {mean_tumor_confidence}")
    if mean_tumor_confidence < min_confidence:
        rejected += 1
        continue

    weight_map = np.ones_like(pred, dtype=np.float32)
    weight_map[pred == 0] = 0.02
    weight_map[pred == 1] = tumor_prob[pred == 1] * 2.0
    weight_map[pred == 2] = pancreas_prob[pred == 2]

    original = sitk.ReadImage(str(img_path))
    out_name = img_path.stem + "_pseudo.mha"

    pseudo_label = sitk.GetImageFromArray(pred)
    pseudo_label.SetSpacing((1.0, 1.0, 1.0))
    pseudo_label.SetOrigin(original.GetOrigin())
    pseudo_label.SetDirection(original.GetDirection())
    sitk.WriteImage(pseudo_label, os.path.join(PSEUDO_LABELS_DIR, out_name))

    pseudo_weight = sitk.GetImageFromArray(weight_map)
    pseudo_weight.SetSpacing((1.0, 1.0, 1.0))
    pseudo_weight.SetOrigin(original.GetOrigin())
    pseudo_weight.SetDirection(original.GetDirection())
    sitk.WriteImage(pseudo_weight, os.path.join(PSEUDO_WEIGHTS_DIR, out_name))

    accepted += 1

    if (i + 1) % 10 == 0:
        print(f"Progress: {i+1}/{len(dataset)} — Accepted: {accepted}, Rejected: {rejected}")

print(f"\nDone!")
print(f"Accepted: {accepted}/{len(dataset)}")
print(f"Rejected: {rejected}/{len(dataset)}")

Model loaded!


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


Founded 367 unlabeled images


  0%|          | 0/367 [00:00<?, ?it/s]Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  0%|          | 1/367 [00:45<4:39:40, 45.85s/it]

Mean confidence tumor: 0.6506319642066956


...

Mean confidence tumor: 0.730958878993988


100%|█████████▉| 366/367 [4:26:13<00:42, 42.59s/it]

Mean confidence tumor: 0.733140766620636


100%|██████████| 367/367 [4:26:55<00:00, 43.64s/it]

Mean confidence tumor: 0.6831401586532593

Done!
Accepted: 166/367
Rejected: 201/367
